# 02. DCA-Trie v1: Static Symbolic Filtering

**Purpose:** Implement and evaluate DCA-Trie v1 — KG path filtering at construction time using the symbolic TypeOracle.

**Changes from GCR baseline:** The trie is built from only those paths that pass both symbolic gates (type gate at terminal hop + range gate at all hops).

**Reference:** Chapter 3, §3.6, Algorithm 1.

**Key difference from old embedding-based design:** No encoder calls, no cosine similarity, no threshold τ. Admission uses TypeOracle symbolic gates.

In [ ]:
# @title 1. Install & Setup
import sys, os, torch, json, re
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from collections import defaultdict

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

!pip install -q transformers==4.44.2 accelerate peft deepspeed tiktoken datasets python-dotenv marisa-trie bitsandbytes trl sentencepiece protobuf wandb networkx

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model
from src.trie import MarisaTrie
import src.utils as utils

# Import symbolic TypeOracle
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from type_oracle import TypeOracle, compute_type_irrelevance

In [ ]:
# @title 2. Configuration
# ═══════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# ═══════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# ═══════════════════════════════════════════════════
# DCA-Trie v1 PARAMETERS (§3.6)
# ═══════════════════════════════════════════════════
# No τ, no encoder. Only symbolic gates.

# ═══════════════════════════════════════════════════
# DECODING
# ═══════════════════════════════════════════════════
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
ATTN_IMPL = "flash_attention_2" if has_a100 else "sdpa"

# ═══════════════════════════════════════════════════
# SAMPLING
# ═══════════════════════════════════════════════════
MAX_SAMPLES = 100          # set to None for full dataset

# ═══════════════════════════════════════════════════
# OUTPUT
# ═══════════════════════════════════════════════════
PREDICT_PATH = "results/GenPaths"
FORCE = True

tag = "DCAv1-symbolic"
print(f"Model: {MODEL_PATH}")
print(f"Dataset: {DATASET}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")
print(f"Tag: {tag}")
print(f"Gates: type_gate (terminal hop) + range_gate (all hops)")

In [ ]:
# @title 3. Load LLM
import argparse
parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading LLM...")
model = LLM(args)
model.prepare_for_inference()
# Suppress sampling warnings
model.generation_cfg.temperature = None
model.generation_cfg.top_p = None
model.model.generation_config.temperature = None
model.model.generation_config.top_p = None
print("Ready.")

In [ ]:
# @title 4. DCA-Trie v1: Build Symbolically Filtered Trie (Algorithm 1)

def build_dcav1_trie_sym(question_dict, tokenizer, index_path_length=2):
    """
    DCA-Trie v1: Static symbolic filtering (Algorithm 1, §3.6.2).

    Steps:
      1. Build TypeOracle from graph triples
      2. Infer answer type constraint from question
      3. Enumerate candidate paths (BFS from question entities)
      4. Filter each path through both symbolic gates:
         - Range gate at every hop (§3.5.6)
         - Type gate at terminal hop only (§3.5.5)
      5. Build trie from surviving paths

    No embeddings, no thresholds, no encoder calls.
    """
    # Step 1: Build oracle from graph
    oracle = TypeOracle.from_graph(question_dict["graph"])

    # Step 2: Infer answer type constraint
    answer_types = oracle.infer_answer_types(question_dict["question"])

    # Step 3: Enumerate candidate paths
    entities = question_dict["q_entity"] if "q_entity" in question_dict else []
    if "paths" in question_dict:
        paths_list = question_dict["paths"]
    else:
        g = utils.build_graph(question_dict["graph"])
        paths_list = utils.dfs(g, entities, index_path_length)

    if len(paths_list) == 0:
        return None, [], None, 0

    # Step 4: Filter by symbolic gates (Algorithm 1 lines 4-17)
    kept_paths = []
    n_type_blocked = 0
    n_range_blocked = 0

    for p in paths_list:
        h = len(p)
        admit = True

        # Gate 2 (range) — must pass at every hop
        for hop_idx, (_, relation, tail_entity) in enumerate(p):
            if not oracle.range_gate(relation, tail_entity):
                n_range_blocked += 1
                admit = False
                break

        # Gate 1 (type) — terminal hop only, on paths that passed range gate
        if admit:
            terminal_entity = p[-1][2]
            if not oracle.type_gate(terminal_entity, answer_types, h, index_path_length):
                n_type_blocked += 1
                admit = False

        if admit:
            kept_paths.append(p)

    if len(kept_paths) == 0:
        return None, [], oracle, n_type_blocked + n_range_blocked

    # Step 5: Build trie
    kept_strs = [utils.path_to_string(p) for p in kept_paths]
    tokenized = tokenizer(kept_strs, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

    return trie, kept_paths, oracle, n_type_blocked + n_range_blocked

In [ ]:
# @title 5. Run DCA-Trie v1 Inference
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")
else:
    print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

def get_output(path, force):
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    with open(path, "r") as f:
        proc = [json.loads(l)["id"] for l in f]
    return open(path, "a"), proc

fout, processed = get_output(os.path.join(output_dir, "predictions.jsonl"), FORCE)

# Accumulators
total_paths_before = 0
total_paths_after = 0
total_type_blocked = 0
total_range_blocked = 0
total_empty = 0

for data in tqdm(dataset, desc="DCA-Trie v1 (Symbolic)"):
    qid = data["id"]
    if qid in processed:
        continue

    # Build DCA-Trie v1 with symbolic filtering
    trie, kept_paths, oracle, n_blocked = build_dcav1_trie_sym(
        data, model.tokenizer, INDEX_LEN
    )

    if trie is None:
        total_empty += 1
        continue

    # Count paths before filtering
    paths_before = len(utils.dfs(
        utils.build_graph(data["graph"]), data["q_entity"], INDEX_LEN
    )) if "paths" not in data else len(data["paths"])
    total_paths_before += paths_before
    total_paths_after += len(kept_paths)

    g = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
    ground_paths = [utils.path_to_string(p) for p in truth_paths]

    # Construct prompt
    prompt_builder = PathGenerationWithAnswerPromptBuilder(
        model.tokenizer, data, model.prefix_prompt
    )
    prompt_text = prompt_builder.build_prompt(
        "zero-shot", data["question"], PROMPT_MODE
    )

    outputs = model.generate(
        prompt_text, data["q_entity"], trie,
        return_ent_cls=False, index_len=INDEX_LEN
    )

    for path_generation in outputs:
        fout.write(json.dumps({
            "id": qid,
            "question": data["question"],
            "prediction": path_generation["prediction"],
            "prediction_score": path_generation.get("score", None),
            "ground_truth": data["answer"],
            "ground_truth_paths": ground_paths,
            "dca_version": "v1",
            "filtering": "symbolic",
            "n_paths_before": paths_before,
            "n_paths_after": len(kept_paths),
        }) + "\n")
    fout.flush()

fout.close()

# Report summary
print("\n" + "=" * 60)
print("FILTERING STATISTICS (Symbolic DCA-Trie v1)")
print("=" * 60)
total_attempted = total_empty + len(dataset) - len(processed) if processed else total_empty + len(dataset)
print(f"Questions with empty trie: {total_empty}")
print(f"Total paths before filtering: {total_paths_before}")
print(f"Total paths after filtering:  {total_paths_after}")
if total_paths_before > 0:
    print(f"Reduction: {(1 - total_paths_after/total_paths_before)*100:.1f}%")
print(f"Total blocked by gates: {total_type_blocked + total_range_blocked}")
print("Done.")

In [ ]:
# @title 6. Per-Gate Breakdown
print("\n" + "=" * 60)
print("PER-GATE FNR ANALYSIS (§3.8)")
print("=" * 60)

# Re-run over the dataset to collect per-gate statistics
val_dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    val_dataset = val_dataset.select(range(min(MAX_SAMPLES, len(val_dataset))))

n_type_false_neg = 0
n_range_false_neg = 0
total_gold_paths = 0
n_questions = 0

for data in tqdm(val_dataset, desc="FNR analysis"):
    g = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
    truth_strs = set(utils.path_to_string(p) for p in truth_paths)
    if len(truth_strs) == 0:
        continue
    total_gold_paths += len(truth_strs)
    n_questions += 1

    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(data["question"])

    for p in truth_paths:
        h = len(p)
        # Check range gate on each hop
        range_fail = False
        for hop_idx, (_, relation, tail_entity) in enumerate(p):
            if not oracle.range_gate(relation, tail_entity):
                range_fail = True
                break

        # Check type gate on terminal hop
        terminal_entity = p[-1][2]
        type_fail = not oracle.type_gate(terminal_entity, answer_types, h, INDEX_LEN)

        if type_fail:
            n_type_false_neg += 1
        if range_fail:
            n_range_false_neg += 1

print(f"Questions with gold paths: {n_questions}")
print(f"Total gold paths: {total_gold_paths}")
print()
print(f"Type gate false negatives:  {n_type_false_neg}  "
      f"(FNR_type = {n_type_false_neg/max(1,total_gold_paths):.4f})")
print(f"Range gate false negatives: {n_range_false_neg}  "
      f"(FNR_range = {n_range_false_neg/max(1,total_gold_paths):.4f})")

# Combined FNR (union bound, Eq. 3.17)
# A gold path is excluded if EITHER gate fails
val_dataset2 = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    val_dataset2 = val_dataset2.select(range(min(MAX_SAMPLES, len(val_dataset2))))

total_excluded = 0
total_gold2 = 0
for data in val_dataset2:
    g = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(data["q_entity"], data["a_entity"], g)
    truth_strs = set(utils.path_to_string(p) for p in truth_paths)
    if len(truth_strs) == 0:
        continue
    total_gold2 += len(truth_strs)

    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(data["question"])

    for p in truth_paths:
        h = len(p)
        excluded = False

        # Range gate
        for hop_idx, (_, relation, tail_entity) in enumerate(p):
            if not oracle.range_gate(relation, tail_entity):
                excluded = True
                break

        # Type gate
        if not excluded:
            terminal_entity = p[-1][2]
            if not oracle.type_gate(terminal_entity, answer_types, h, INDEX_LEN):
                excluded = True

        if excluded:
            total_excluded += 1

print()
print(f"Combined FNR (either gate fails): "
      f"{total_excluded/max(1,total_gold2):.4f}")
print(f"Target: FNR < 0.05 (Condition P, §3.4.1)")

In [ ]:
# @title 7. Compare with GCR Baseline
print("""
=== Comparison: GCR vs DCA-Trie v1 (Symbolic) ===

Metric               GCR         DCA-Trie v1 (symbolic)
──────               ───         ─────────────────────
Hits@1               89.0 (dev)  ?
F1                   ?           ?
Faithfulness        100%        100% (unchanged — Condition F)
Avg trie size        ~5000       ? (should be lower — pruned by gates)
SIR*                 high        ? (should be lower)
SIR*_type            high        ~0 (type gate kills these)
SIR*_traj            high        ? (range gate kills these)

To measure decomposed SIR, run notebook 04_SIR_Evaluation.ipynb
on the predictions.jsonl produced above.
""")